In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [11]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [12]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [13]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(dim=1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6], device='cuda:0')


# Model Layers
A sample minibatch of 3 images of size 28x28

In [14]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


# nn.Flatten
We initialize the nn.Flatten layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values ( the minibatch dimension (at dim=0) is maintained).

In [15]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


# nn.Linear
The linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

In [16]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


# nn.ReLU
Non-linear activations are what create the complex mappings between the model’s inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In [17]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1701,  0.0575,  0.1118,  0.5045, -0.1695, -0.1562, -0.0186, -0.1517,
         -0.3190,  0.4338,  0.6335,  0.0811, -0.0696,  0.1142,  0.3782, -0.4143,
         -0.3911,  0.2585, -0.5725, -0.1366],
        [-0.3413,  0.2415, -0.2181,  0.3437, -0.0013, -0.0798,  0.1461,  0.1639,
         -0.2082,  0.1340,  0.6630,  0.2538, -0.2014,  0.1642,  0.2669,  0.0164,
         -0.4323,  0.1303, -0.2641, -0.1029],
        [-0.1317,  0.6187, -0.0080, -0.2083,  0.1113, -0.5371,  0.4716, -0.1321,
         -0.2190,  0.1509,  0.5860, -0.0859,  0.3016,  0.0844,  0.1273, -0.1924,
         -0.0200,  0.2141, -0.4875, -0.1699]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0575, 0.1118, 0.5045, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.4338, 0.6335, 0.0811, 0.0000, 0.1142, 0.3782, 0.0000, 0.0000, 0.2585,
         0.0000, 0.0000],
        [0.0000, 0.2415, 0.0000, 0.3437, 0.0000, 0.0000, 0.1461, 0.1639, 0.0000,
         0.1340, 0.6630, 0.2538, 0.0000, 0.1642, 0.26

# nn.Sequential
nn.Sequential is an ordered container of modules. The data is passed through all the modules in the same order as defined. Sequential containers can be used to put together a quick network like seq_modules.

In [18]:
seq_modules = nn.Sequential(
    flatten, 
    layer1, 
    nn.ReLU(),
    nn.Linear(20, 10)
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)

# nn.Softmax
The last linear layer of the neural network returns logits - raw values in [-infty, infty] - which are passed to the nn.Softmax module. The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class. dim parameter indicates the dimension along which the values must sum to 1.

In [19]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

# Model Parameters
Many layers inside a neural network are parameterized, i.e. have associated weights and biases that are optimized during training. Subclassing nn.Module automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model’s parameters() or named_parameters() methods.

In this example, we iterate over each parameter, and print its size and a preview of its values

In [20]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]}\n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0179,  0.0241,  0.0095,  ...,  0.0240,  0.0179,  0.0186],
        [-0.0143, -0.0221,  0.0239,  ...,  0.0179, -0.0243,  0.0112]],
       device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0045,  0.0084], device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0362, -0.0003,  0.0223,  ...,  0.0294,  0.0162, -0.0111],
        [-0.0367, -0.0191,  0.0235,  ..., -0.0135, -0.0103,  0.0436]],
       device='cuda:0', grad_fn=<Slic